In [277]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
import requests

from dotenv import load_dotenv

load_dotenv()

True

In [278]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [279]:
import os

EXCHANGE_RATE_API_KEY= os.getenv('EXCHANGE_RATE_API_KEY')

print(EXCHANGE_RATE_API_KEY)

566d2383234154630a5961a9


In [280]:
# tools creation

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """

    url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/pair/{base_currency}/{target_currency}'

    response = requests.get(url)
    return response.json()


In [281]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1787961601,
 'time_last_update_utc': 'Sat, 29 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1788048001,
 'time_next_update_utc': 'Sun, 30 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.5338}

In [282]:
@tool
def convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_currency_value * conversion_rate

In [283]:
convert.invoke({'base_currency_value':10, 'conversion_rate':95})

950.0

### Binding the tool with LLM

In [284]:
llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-72B-Instruct',
    task='text-generation'
)
model = ChatHuggingFace(llm=llm)

llm_with_tools = model.bind_tools([get_conversion_factor, convert])

In [285]:
messages = [HumanMessage('What is the conversion factor between usd and inr. And what is the conversion factor between USD and GBP. Can you convert 20 GBP to INR')]

In [286]:
messages

[HumanMessage(content='What is the conversion factor between usd and inr. And what is the conversion factor between USD and GBP. Can you convert 20 GBP to INR', additional_kwargs={}, response_metadata={})]

In [287]:
ai_message = llm_with_tools.invoke(messages)
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_TyYJZbHNN6C7VPC7JaQpt8yj', 'type': 'function'}, {'function': {'arguments': '{"base_currency": "USD", "target_currency": "GBP"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_vkvJpaJejsKgj43IEj3xahVB', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 20}', 'name': 'convert', 'description': None}, 'id': 'call_IxBSaiA7sf3JX0zbSXycHb0z', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 453, 'total_tokens': 534}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04e54-bccb-7eb1-b75d-624c0bfb09c5-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '

In [288]:
messages.append(ai_message)

In [289]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_TyYJZbHNN6C7VPC7JaQpt8yj',
  'type': 'tool_call'},
 {'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'GBP'},
  'id': 'call_vkvJpaJejsKgj43IEj3xahVB',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 20},
  'id': 'call_IxBSaiA7sf3JX0zbSXycHb0z',
  'type': 'tool_call'}]

In [290]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        #fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to messages list
        messages.append(tool_message1)
    # execute the 2nd took using the conversion rate from tool 1
    if tool_call['name'] == 'convert':
        #fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

   

In [291]:
messages

[HumanMessage(content='What is the conversion factor between usd and inr. And what is the conversion factor between USD and GBP. Can you convert 20 GBP to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_TyYJZbHNN6C7VPC7JaQpt8yj', 'type': 'function'}, {'function': {'arguments': '{"base_currency": "USD", "target_currency": "GBP"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_vkvJpaJejsKgj43IEj3xahVB', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 20}', 'name': 'convert', 'description': None}, 'id': 'call_IxBSaiA7sf3JX0zbSXycHb0z', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 453, 'total_tokens': 534}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': None, 'finish_r

In [292]:
llm_with_tools.invoke(messages)

AIMessage(content='The conversion factor between USD and INR is 95.5338, and the conversion factor between USD and GBP is 0.7379. \n\nTo convert 20 GBP to INR, we first need to convert 20 GBP to USD, and then convert the resulting USD amount to INR. \n\nGiven that 20 GBP is approximately 14.758 USD (using the conversion rate of 0.7379), and the conversion rate from USD to INR is 95.5338, the final conversion from 20 GBP to INR is:\n\n20 GBP = 14.758 USD * 95.5338 = 1409.96 INR.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 1027, 'total_tokens': 1192}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04e54-e18f-7443-8c2b-1f62c6513d92-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1027, 'output_tokens': 165, 'total_tokens': 1192})